In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "InitialFigures")
dataType = "CompareSkinTemperature"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "WET"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

spinup_hours = 24
# spinup_hours = 12

# RunType = ("TRACER","WET","NSSL",spinup_hours)
RunType = ("TRACER","DRY","NSSL",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

# RunType = ("TRACER","WET","TEMPO",spinup_hours)
RunType = ("TRACER","DRY","TEMPO",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
####################################
#OBSERVATION DATA LOADING

In [ ]:
# gndirt
# Description: Infrared Thermometer: Ground surface temperature
# Site: Houston, TX; Tracking Aerosol Convection interactions ExpeRiment (HOU)
# Location: Houston, TX; AMF1 (main site for TRACER) 
# Facility Code: M1
# Category: Radiometric
# Data Type: Routine Data 
# Source Instrument/Data: Infrared Thermometer 
# Start Date: 2021-08-04 
# End Date: 2022-10-01 

In [ ]:
#getting dataPath
def GetDataFolder(dataClassification,region,dataFolderName):
    dataPath = os.path.join(DirectoryManager.dataDirectory,dataClassification,region,dataFolderName)
    return dataPath
    
dataClassification = "Observation_Data"
region = ModelData_NSSL.region
dataFolderName = "TRACER_GNDirt"
dataPath = GetDataFolder(dataClassification,region,dataFolderName)

#reading data
fileList, filePathList = DirectoryManager.ListFiles(dataPath)
# xr.open_dataset(filePathList[5])["sfc_ir_temp"].plot()

In [ ]:
target_dates = ["20220630", "20220701", "20220702"]
ids = [i for i, f in enumerate(fileList) if any(date in f for date in target_dates)]

surface_data = []; surface_time = []
for count,i in enumerate(ids):
    currentFilePath = filePathList[i]
    
    # Open dataset and extract variable
    ds = xr.open_dataset(currentFilePath)
    var = ds["sfc_ir_temp"]
    if count == 0:
        surfaceDataLat = ds['lat'].item()
        surfaceDataLon = ds['lon'].item()
    
    # Append data and time
    surface_data.append(var.data)
    surface_time.append(var["time"].data)

# --- Combine all data and time ---
surface_data = np.concatenate(surface_data)
surface_time = np.concatenate(surface_time)

In [ ]:
####################################
#MODEL DATA LOADING

In [ ]:
def findNearestTimes(reference_times, times):
    """
    For each time in surface_time, find the index of the closest time in model_times.
    """
    
    # Use broadcasting to find the absolute difference and take the argmin
    idx = np.abs(reference_times[:, None] - times[None, :]).argmin(axis=1)
    
    return idx.tolist()

time_strings = [t.replace(":", ".") for t in ModelData_NSSL.timeStrings]
model_times = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]
model_times = np.array(model_times, dtype='datetime64[ns]')
closest_times = findNearestTimes(reference_times=surface_time, times=model_times)

In [ ]:
def GetDataTimestep_cached(ModelData, t, varName, cache):
    """
    Retrieve model variable for a given timestep using an in-memory cache.
    """
    # If time already loaded, return from cache
    if t in cache:
        return cache[t]
    else:
        print(f"loading for time {t}")
    
    # Otherwise, load and store it
    data = ModelData.GetDataTimestep(t=t, varName=varName)
    cache[t] = data
    return data


In [ ]:
import os
import pickle
import numpy as np

def GetModelSurfaceData(ModelData, closest_times, varName, 
                                   lat, lon, inputDirectory):
    """
    Loads model surface data if cached, otherwise computes and saves it.
    """

    # --- 1. If pickle file exists, load it ---
    if os.path.exists(inputDirectory):
        print(f"Loading cached model surface data from {inputDirectory}")
        with open(inputDirectory, "rb") as f:
            modelSurfaceData = pickle.load(f)
        return np.array(modelSurfaceData)

    # --- 2. Otherwise, compute and save ---
    print("Cache not found — computing model surface data...")
    time_cache = {}
    modelSurfaceData = []

    for t in closest_times:
        data_t = GetDataTimestep_cached(ModelData, t=t, varName=varName, cache=time_cache)
        selection = data_t.sel(latitude=lat, longitude=lon, method="nearest").data
        modelSurfaceData.append(selection)

    # Convert to NumPy array for easier later use
    modelSurfaceData = np.array(modelSurfaceData)

    # --- 3. Save to pickle ---
    with open(inputDirectory, "wb") as f:
        pickle.dump(modelSurfaceData, f)
    print(f"Saved computed model surface data to {inputDirectory}")

    return modelSurfaceData


In [ ]:
#loading for NSSL
ModelData=ModelData_NSSL

modelSurfaceData_NSSL = GetModelSurfaceData(
    ModelData=ModelData,
    closest_times=closest_times,
    varName="skintemp",           # better to use "skintemp"
    lat=surfaceDataLat,
    lon=surfaceDataLon,
    inputDirectory=f"modelSurfaceData_skintemp_{ModelData.region}_{ModelData.case}_{ModelData.mpType}.pkl"
)

In [ ]:
#loading for TEMPO
ModelData=ModelData_TEMPO

modelSurfaceData_TEMPO = GetModelSurfaceData(
    ModelData=ModelData,
    closest_times=closest_times,
    varName="skintemp",
    lat=surfaceDataLat,
    lon=surfaceDataLon,
    inputDirectory=f"modelSurfaceData_skintemp_{ModelData.region}_{ModelData.case}_{ModelData.mpType}.pkl"
)

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
import matplotlib.dates as mdates
def PlotSurfaceComparison(surface_time, surface_data,
                          model_time, model_data,
                          variable_name="Surface Temperature", units="K",
                          site_name="Houston (TRACER)", model_label="MPAS"):
    """
    Create a clean time series comparison plot between observed and one or more model datasets.

    Works seamlessly when:
      - model_data is a single array OR a list of arrays
      - model_time is a single array (replicated for all models)
      - model_label is a single string OR a list of strings

    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure handle for saving or further customization.
    ax : matplotlib.axes.Axes
        Axis handle for additional plotting.
    """

    fig, ax = plt.subplots(figsize=(10, 5))

    # --- Plot observations ---
    ax.plot(surface_time, surface_data, color='black', linewidth=1.8,
            label=f"Observation ({site_name})")

    # --- Normalize model inputs ---
    if not isinstance(model_data, (list, tuple)):
        model_data = [model_data]
    if not isinstance(model_time, (list, tuple)):
        model_time = [model_time] * len(model_data)
    if isinstance(model_label, str):
        model_label = [model_label] * len(model_data)

    if not (len(model_data) == len(model_time) == len(model_label)):
        raise ValueError("Lengths of model_data, model_time, and model_label must match.")

    # --- Plot models ---
    colors = plt.cm.tab10.colors
    for i, (m_time, m_data, label) in enumerate(zip(model_time, model_data, model_label)):
        ax.plot(m_time, m_data, color=colors[i % len(colors)],
                linewidth=1.6, linestyle='-', label=f"Model ({label})")

    # --- Labels and formatting ---
    ax.set_title(f"{variable_name} Comparison", fontsize=14, weight='bold')
    ax.set_xlabel("Time (UTC)", fontsize=12)
    ax.set_ylabel(f"{variable_name} [{units}]", fontsize=12)

    # Format time axis nicely
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
    fig.autofmt_xdate()

    # --- Grid, legend, and style tweaks ---
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(frameon=False, fontsize=10, loc='upper left')
    fig.tight_layout()

    return fig, ax


In [ ]:
def SaveFigure(fig, dataType, dpi=150, extension="png"):
    """
    Saves a matplotlib Figure to a subdirectory named after the model configuration.
    """

    # Ensure dpi is a plain Python int
    dpi = int(np.atleast_1d(dpi)[0])  # Handles np.float64 or array inputs safely

    # --- Define output subdirectory ---
    outputSubDirectory = f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType}vs{ModelData_TEMPO.mpType}_{ModelData_NSSL.spinup_hours}hrs"
    save_dir = os.path.join(outputPlottingDirectory, outputSubDirectory)
    os.makedirs(save_dir, exist_ok=True)

    # --- File path ---
    outputFile = os.path.join(
        save_dir,
        f"{dataType}.{extension}"
    )

    # --- Save and close ---
    fig.savefig(outputFile, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure to: {outputFile}")

In [ ]:
####################################
#PLOTTING

In [ ]:
fig,ax = PlotSurfaceComparison(
    surface_time=surface_time,
    surface_data=surface_data,
    model_time=surface_time, 
    model_data=[modelSurfaceData_NSSL,modelSurfaceData_TEMPO],
    variable_name="Ground Skin Temperature",
    units="K",
    site_name="TRACER AMF1 GNDIRT Data - Ground Surface Temperature",
    model_label=[f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_NSSL",f"{ModelData_NSSL.region}_{ModelData_NSSL.case}_TEMPO"]
)
SaveFigure(fig, dataType)